<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting & Cleaning

## Setup

In [ ]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from shapely.geometry import shape
import requests
from google.colab import userdata
from google.colab import drive
from pprint import pprint as print

In [2]:
# Mount Google Drive
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [3]:
# Set up the API
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
}

In [4]:
# Fetching a sample of data from the right time period
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

[{'count': '11707067'}]


In [ ]:
# Collecting all the taxi data within the time period specified, if already collected use option 2
def load_raw_data(option = 1):
  if option == 1:
    query = """
    SELECT
      *
    WHERE
      tpep_pickup_datetime >= "2025-06-21T00:00:00"
      AND
      tpep_dropoff_datetime <= "2025-09-21T23:59:59"
    LIMIT 12000000
    """

    query_payload = {
        "query": query
    }

    query_response = requests.get(data_api, headers=headers, params=query_payload)
    print("All Done!!!")
    query_data = query_response.json()
    taxi_df = pd.DataFrame(query_data)
    return taxi_df
  elif option == 2:
    taxi_df = pd.read_csv("/content/drive/MyDrive/raw_taxi_summer_data.csv")
    return taxi_df
  else:
    print("Invalid option")
    return None

In [6]:
taxi_df:pd.DataFrame = load_raw_data(2)

## Data Structure

In [7]:
# Preview the first few rows using the dataframe head function
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at
0,2,2025-06-21 00:00:00,2025-06-21 00:07:21,1.0,1.17,1.0,137,113,1,9.30,...,0.0,1.0,16.71,2.5,0.0,0.75,row-emei~byqv-hzqe,rv-dnr4~ztm9.kep3,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z
1,2,2025-06-21 00:00:00,2025-06-21 00:04:21,2.0,0.61,1.0,79,107,1,5.80,...,0.0,1.0,13.86,2.5,0.0,0.75,row-h2b5-wa6w_t924,rv-qyaa~vq94-iwvb,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z
2,2,2025-06-21 00:00:00,2025-06-21 00:08:00,NaN,1.70,NaN,74,42,0,-1.50,...,0.0,1.0,2.17,NaN,NaN,0.00,row-xpsy~jicp~dcpm,rv-awpk_x4fp~ci3q,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z
3,2,2025-06-21 00:00:00,2025-06-21 00:29:45,NaN,12.49,NaN,261,244,0,43.43,...,0.0,1.0,48.18,NaN,NaN,0.75,row-9xbz-xfrr_zane,rv-efyf~7pu6_7u7f,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z
4,2,2025-06-21 00:00:00,2025-06-21 00:11:00,NaN,1.73,NaN,50,162,0,16.26,...,0.0,1.0,25.21,NaN,NaN,0.75,row-fqhg.5ceh~usg7,rv-xevg~cyge.7u66,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z


In [8]:
# Preview the last few rows using the dataframe head function
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at
11707062,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,186,234,1,6.5,...,0.0,1.0,14.70,2.5,0.0,0.75,row-29ph~gzpu_tazy,rv-xkhe.6g4x-xzhz,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z
11707063,2,2025-09-21 23:58:15,2025-09-21 23:58:20,2.0,0.00,5.0,265,265,1,66.0,...,0.0,1.0,67.00,0.0,0.0,0.00,row-4g6k~vzic_7j3y,rv-zkyc-739x~wjtu,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z
11707064,2,2025-09-21 23:58:26,2025-09-21 23:58:30,1.0,0.05,5.0,265,265,1,150.0,...,0.0,1.0,151.00,0.0,0.0,0.00,row-jig5-i2j5_wxsw,rv-hack~bkni~b43c,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z
11707065,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,48,179,1,24.7,...,0.0,1.0,30.45,2.5,0.0,0.75,row-mt2k.x3q7~ypn4,rv-ca5i-jvv7.n7tt,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z
11707066,7,2025-09-21 23:59:43,2025-09-21 23:59:43,1.0,1.03,1.0,249,107,1,9.3,...,0.0,1.0,18.81,2.5,0.0,0.75,row-hgds.bskj~drqf,rv-fknd-iyz9~ecyd,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z


In [9]:
# Displaying the Dataframes shape, columns, and amount of non null values using the info function
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707067 entries, 0 to 11707066
Data columns (total 23 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   vendorid               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   ratecodeid             float64
 6   pulocationid           int64  
 7   dolocationid           int64  
 8   payment_type           int64  
 9   fare_amount            float64
 10  extra                  float64
 11  mta_tax                float64
 12  tip_amount             float64
 13  tolls_amount           float64
 14  improvement_surcharge  float64
 15  total_amount           float64
 16  congestion_surcharge   float64
 17  airport_fee            float64
 18  cbd_congestion_fee     float64
 19  :id                    object 
 20  :version               object 
 21  :created_at            object 
 22  :updated_at     

In [22]:
# Drop columns not needed for the project
taxi_df.drop(columns=['request_source', 'store_and_fwd_flag',':id',':version',':created_at',':updated_at'], axis=1, inplace=True)

In [11]:
# Converting our two datetime columns to the pandas datetime type
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

## Missing and Duplicate Data

In [12]:
# Remove duplicate records from the dataset
taxi_df = taxi_df.drop_duplicates().reset_index(drop=True)

In [13]:
# collecting numeric columns to handle missing values
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [14]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

,0
vendorid,0.000000
tpep_pickup_datetime,0.000000
tpep_dropoff_datetime,0.000000
passenger_count,26.594244
trip_distance,0.000000
ratecodeid,26.594244
pulocationid,0.000000
dolocationid,0.000000
payment_type,0.000000
fare_amount,0.000000


In [15]:
# Inspecting the ratecodeid for its values
print(taxi_df['ratecodeid'].value_counts(dropna=False))

ratecodeid
1.0     8059782
NaN     3113406
2.0      329229
5.0      126872
3.0       41044
4.0       34420
99.0       2306
6.0           8
Name: count, dtype: int64


In [16]:
# Fill missing values in ratecodeid
taxi_df['ratecodeid'] = taxi_df['ratecodeid'].fillna(99)

In [17]:
# Inspecting the congestion_surcharge for its values
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

congestion_surcharge
 2.50    7687339
 NaN     3113406
 0.00     727295
-2.50     179021
 1.00          5
 0.75          1
Name: count, dtype: int64


In [18]:
# Handiling congestion surchage values by replacing it with a boolean indicator
taxi_df['congestion_surcharge'].replace(
    {
      2.50: 1,
      0.00: 0,
      -2.50: 1,
      1.00: 1,
      0.75: 1
    },
    inplace=True
)

taxi_df['congestion_surcharge'] = taxi_df['congestion_surcharge'].fillna(-1)

/tmp/ipykernel_17545/305008353.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['congestion_surcharge'].replace(


In [19]:
# Inspecting the airport_fee for its values
print(taxi_df['airport_fee'].value_counts(dropna=False))

airport_fee
 0.00    7700775
 NaN     3113406
 1.75     834200
-1.75      52905
 6.75       4535
 5.00       1245
 0.50          1
Name: count, dtype: int64


In [20]:
# Handiling airport_fee values by replacing it with a boolean indicator
taxi_df['airport_fee'].replace(
    {
      0.00: 0,
      1.75: 1,
      -1.75: 1,
      6.75: 1,
      5.00: 1,
      0.50: 1
    },
    inplace=True
)

taxi_df['airport_fee'] = taxi_df['airport_fee'].fillna(-1)

/tmp/ipykernel_17545/3961128368.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['airport_fee'].replace(


In [25]:
# checking column values of when passenger count is empty
empty_passenger = taxi_df.loc[taxi_df['passenger_count'].isna()]
pmissing = (empty_passenger.isna().sum() / len(empty_passenger)) * 100
pmissing

,0
vendorid,0.0
tpep_pickup_datetime,0.0
tpep_dropoff_datetime,0.0
passenger_count,100.0
trip_distance,0.0
ratecodeid,0.0
pulocationid,0.0
dolocationid,0.0
payment_type,0.0
fare_amount,0.0


## Taxi Zones Data

In [26]:
# Querying for taxi zones data
zones_query = {
    "query":"""
    SELECT
      *
    LIMIT 500
    """
}
zones_response = requests.get(
    "https://data.cityofnewyork.us/api/v3/views/8meu-9t5y/query.json",
    headers=headers,
    params=zones_query
  )
zone_data = zones_response.json()
print("All done!")

'All done!'


In [27]:
zone_df = pd.DataFrame(zone_data)
zone_df["the_geom"] = zone_df["the_geom"].apply(lambda g: shape(g) if isinstance(g, dict) else None)
zone_df = geopandas.GeoDataFrame(zone_df, geometry="the_geom", crs="EPSG:4326")

In [28]:
# Remove metadata columns
zone_df.drop(columns=[':id', ':version', ':created_at', ':updated_at'], axis=1, inplace=True)

In [29]:
# fixing datatypes of three specific columns specified below
zone_df['shape_leng'] = zone_df['shape_leng'].astype("float")
zone_df['shape_area'] = zone_df['shape_area'].astype("float")
zone_df['locationid'] = zone_df['locationid'].astype("int")

In [30]:
# checking our dataframes structure again
zone_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   the_geom    263 non-null    geometry
 1   shape_leng  263 non-null    float64 
 2   shape_area  263 non-null    float64 
 3   zone        263 non-null    object  
 4   locationid  263 non-null    int64   
 5   borough     263 non-null    object  
dtypes: float64(2), geometry(1), int64(1), object(2)
memory usage: 12.5+ KB


## Borough Validation

In [31]:
# importing datasets from Google Drive
taxi_df = pd.read_csv("/content/drive/MyDrive/taxi_summer_data.csv")
zone_df = geopandas.read_file("/content/drive/MyDrive/taxi_zones_data.json")

In [ ]:
# merging out tables
taxi_zone_df = pd.merge(taxi_df, zone_df, left_on='pulocationid', right_on='locationid', how='left')
taxi_zone_df = pd.merge(taxi_zone_df, zone_df, left_on='dolocationid', right_on='locationid', how='left')
taxi_zone_df.rename(columns={'borough_x': 'pickup_borough', 'borough_y': 'dropoff_borough'}, inplace=True)
taxi_zone_df.drop(columns=['locationid_x', 'locationid_y'], inplace=True)
taxi_zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11719616 entries, 0 to 11719615
Data columns (total 28 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   tpep_pickup_datetime   object  
 1   tpep_dropoff_datetime  object  
 2   passenger_count        float64 
 3   trip_distance          float64 
 4   ratecodeid             float64 
 5   pulocationid           int64   
 6   dolocationid           int64   
 7   payment_type           int64   
 8   fare_amount            float64 
 9   extra                  float64 
 10  mta_tax                float64 
 11  tip_amount             float64 
 12  tolls_amount           float64 
 13  improvement_surcharge  float64 
 14  total_amount           float64 
 15  congestion_surcharge   float64 
 16  airport_fee            float64 
 17  cbd_congestion_fee     float64 
 18  shape_leng_x           float64 
 19  shape_area_x           float64 
 20  zone_x                 object  
 21  pickup_borough         object

In [ ]:
# checking pickup borough values
taxi_zone_df['pickup_borough'].value_counts(dropna=False)

,count
pickup_borough,
Manhattan,9929492
Queens,1251051
Brooklyn,440871
Bronx,70319
NaN,25077
EWR,1652
Staten Island,1154


In [ ]:
# checking dropoff borough values
taxi_zone_df['dropoff_borough'].value_counts(dropna=False)

,count
dropoff_borough,
Manhattan,9983606
Queens,774378
Brooklyn,710326
Bronx,138611
NaN,86475
EWR,22357
Staten Island,3863


In [ ]:
# checking pickup borough values when NaN 
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

,pulocationid,pickup_borough,dolocationid,dropoff_borough
204,264,NaN,162,Manhattan
1112,265,NaN,265,NaN
1113,265,NaN,265,NaN
1376,265,NaN,265,NaN
1497,264,NaN,264,NaN
...,...,...,...,...
11719529,265,NaN,265,NaN
11719573,265,NaN,265,NaN
11719605,265,NaN,265,NaN
11719612,265,NaN,265,NaN


In [36]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['pickup_borough'].isna(), 'pulocationid'].value_counts()


,count
pulocationid,
264,17933
265,6679
57,464
105,1


In [ ]:
# checking pickup borough values when NaN
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), ['pulocationid', 'pickup_borough', 'dolocationid','dropoff_borough']]

,pulocationid,pickup_borough,dolocationid,dropoff_borough
102,100,Manhattan,57,NaN
273,237,Manhattan,265,NaN
278,234,Manhattan,265,NaN
439,249,Manhattan,264,NaN
961,263,Manhattan,264,NaN
...,...,...,...,...
11719529,265,NaN,265,NaN
11719573,265,NaN,265,NaN
11719605,265,NaN,265,NaN
11719612,265,NaN,265,NaN


In [38]:
# value counts of location id where borough is null
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'].isna(), 'dolocationid'].value_counts()

,count
dolocationid,
265,64608
264,21278
57,588
105,1


In [39]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['pickup_borough'] == 'EWR', 'pulocationid'].value_counts()


,count
pulocationid,
1,1652


In [40]:
# value counts of location id where borough is ewr
taxi_zone_df.loc[taxi_zone_df['dropoff_borough'] == 'EWR', 'dolocationid'].value_counts()

,count
dolocationid,
1,22357


In [41]:
# dropping taxi data where location id is specific values 265, 264, 57, 105, 1
taxi_df = taxi_df.loc[~taxi_df['pulocationid'].isin([265, 264, 57, 105, 1])]
taxi_df = taxi_df.loc[~taxi_df['dolocationid'].isin([265, 264, 57, 105, 1])]
taxi_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 11587258 entries, 0 to 11707065
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   tpep_pickup_datetime   object 
 1   tpep_dropoff_datetime  object 
 2   passenger_count        float64
 3   trip_distance          float64
 4   ratecodeid             float64
 5   pulocationid           int64  
 6   dolocationid           int64  
 7   payment_type           int64  
 8   fare_amount            float64
 9   extra                  float64
 10  mta_tax                float64
 11  tip_amount             float64
 12  tolls_amount           float64
 13  improvement_surcharge  float64
 14  total_amount           float64
 15  congestion_surcharge   float64
 16  airport_fee            float64
 17  cbd_congestion_fee     float64
dtypes: float64(13), int64(3), object(2)
memory usage: 1.6+ GB


In [ ]:
# double checking ids
taxi_df.loc[taxi_df['pulocationid'].isin([265, 264, 57, 105, 1])]

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee


In [ ]:
# double checking ids
taxi_df.loc[taxi_df['dolocationid'].isin([265, 264, 57, 105, 1])]

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee


In [44]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

,0
tpep_pickup_datetime,0.000000
tpep_dropoff_datetime,0.000000
passenger_count,26.777008
trip_distance,0.000000
ratecodeid,0.000000
pulocationid,0.000000
dolocationid,0.000000
payment_type,0.000000
fare_amount,0.000000
extra,0.000000


## Export

In [45]:
# Export the processed taxi records and spatial zone lookup tables to Google Drive
taxi_df.to_csv("/content/drive/MyDrive/taxi_summer_data.csv", index=False)
zone_df.to_file("/content/drive/MyDrive/taxi_zones_data.json", index=False)